In [ ]:
# # install packages
# !pip install -qq matpower matpowercaseframes

In [ ]:
from importlib.metadata import version

import matpower

print(f"matpower version: {matpower.__version__}")
print(f"matlabengine version: {version('matlabengine')}")

matpower version: 8.1.0.2.2.4
matlabengine version: 25.2.2


In [ ]:
from matpower import start_instance

In [ ]:
m = start_instance(engine="matlab")  # runpf require oct2py / matlab

## `runpf` default to `case9.m`

Note:
matlabengine does not support sparse matrix, making runpf result unpassable to python.

In [ ]:
m.runpf(nargout=0)  # require nargout to avoid TypeError due to sparse


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Power Flow -- AC-polar-power formulation

Newton's method converged in 4 iterations.
PF successful

Converged in 0.33 seconds
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    319.6              22.8
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)             -0.0               0.0
Branches           9     Losses (I^2 * Z)         4.64             48.38
Transforme

## `runpf` from `mpc` loaded using `loadcase`

In [ ]:
mpc = m.loadcase("case9")
m.runpf(mpc, nargout=0)


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Power Flow -- AC-polar-power formulation

Newton's method converged in 4 iterations.
PF successful

Converged in 0.11 seconds
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    319.6              22.8
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)             -0.0               0.0
Branches           9     Losses (I^2 * Z)         4.64             48.38
Transforme

In [ ]:
mpc.keys()

dict_keys(['version', 'baseMVA', 'bus', 'gen', 'branch', 'gencost'])

In [ ]:
mpc["gencost"]

matlab.double([[2.0,1500.0,0.0,3.0,0.11,5.0,150.0],[2.0,2000.0,0.0,3.0,0.085,1.2,600.0],[2.0,3000.0,0.0,3.0,0.1225,1.0,335.0]])

## `runpf` wrapper to avoid `TypeError`

In [ ]:
def runpf(mpc, m=None, inplace=True):
    if m is None:
        m = start_instance(engine="matlab")
        SHUTDOWN = True
    else:
        SHUTDOWN = False

    m.workspace["mpc_"] = mpc
    m.eval("r1_ = runpf(mpc_);", nargout=0)

    if not inplace:
        mpc = {}

    _extract_result(mpc, m)

    if SHUTDOWN:
        m.quit()

    return mpc


def _extract_result(mpc, m):
    mpc["baseMVA"] = m.eval("r1_.baseMVA;", nargout=1)
    mpc["version"] = m.eval("r1_.version;", nargout=1)
    mpc["bus"] = m.eval("r1_.bus;", nargout=1)
    mpc["gen"] = m.eval("r1_.gen;", nargout=1)
    mpc["branch"] = m.eval("r1_.branch;", nargout=1)
    mpc["gencost"] = m.eval("r1_.gencost;", nargout=1)

In [ ]:
mpc = m.loadcase("case9")
mpc = runpf(mpc)


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Power Flow -- AC-polar-power formulation

Newton's method converged in 4 iterations.
PF successful

Converged in 0.34 seconds
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    319.6              22.8
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)             -0.0               0.0
Branches           9     Losses (I^2 * Z)         4.64             48.38
Transforme

In [ ]:
mpc

{'version': '2',
 'baseMVA': 100.0,
 'bus': matlab.double([[1.0,3.0,0.0,0.0,0.0,0.0,1.0,1.04,0.0,345.0,1.0,1.1,0.9],[2.0,2.0,0.0,0.0,0.0,0.0,1.0,1.025,9.280005481642794,345.0,1.0,1.1,0.9],[3.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0250000000000001,4.664751333136763,345.0,1.0,1.1,0.9],[4.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0257883928440104,-2.2167877999497896,345.0,1.0,1.1,0.9],[5.0,1.0,90.0,30.0,0.0,0.0,1.0,1.0126543240177757,-3.6873961701570614,345.0,1.0,1.1,0.9],[6.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0323529490023684,1.9667160744490775,345.0,1.0,1.1,0.9],[7.0,1.0,100.0,35.0,0.0,0.0,1.0,1.015882583627499,0.7275360768742919,345.0,1.0,1.1,0.9],[8.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0257693723864543,3.7197011546217573,345.0,1.0,1.1,0.9],[9.0,1.0,125.0,50.0,0.0,0.0,1.0,0.9956308580482947,-3.988805272851468,345.0,1.0,1.1,0.9]]),
 'gen': matlab.double([[1.0,71.64102147448236,27.045923533492328,300.0,-300.0,1.04,100.0,1.0,250.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[2.0,163.0,6.653660318427293,300.0,-300.0,1.025,10

## `runopf` require tricks to avoid `<object opf_model>` 

In [ ]:
mpc = m.loadcase("case9")
m.runopf(mpc, nargout=0)


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Optimal Power Flow -- AC-polar-power formulation
MATPOWER Interior Point Solver -- MIPS, Version 1.5.2, 12-Jul-2025
 (using built-in linear solver)
Converged!
OPF successful

Converged in 0.31 seconds
Objective Function Value = 5296.69 $/hr
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    318.3              -9.6
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)    

Using `m.runopf(..., nout='max_nout')` will trigger matpower to use:

```octave
[baseMVA, bus, gen, gencost, branch, f, success, et] = runopf(...);
```

You can see matpower `runopf()` docs using:

```ipython
m.runopf?
```

## `runopf` wrapper

In [ ]:
def runopf(mpc, m=None, inplace=True):
    if m is None:
        m = start_instance(engine="matlab")
        SHUTDOWN = True
    else:
        SHUTDOWN = False

    m.workspace["mpc_"] = mpc
    m.eval("r1_ = runopf(mpc_);", nargout=0)

    if not inplace:
        mpc = {}

    _extract_result(mpc, m)

    if SHUTDOWN:
        m.quit()

    return mpc


def _extract_result(mpc, m):
    mpc["baseMVA"] = m.eval("r1_.baseMVA;", nargout=1)
    mpc["version"] = m.eval("r1_.version;", nargout=1)
    mpc["bus"] = m.eval("r1_.bus;", nargout=1)
    mpc["gen"] = m.eval("r1_.gen;", nargout=1)
    mpc["branch"] = m.eval("r1_.branch;", nargout=1)
    mpc["gencost"] = m.eval("r1_.gencost;", nargout=1)

In [ ]:
mpc = m.loadcase("case9")
mpc = runopf(mpc, m=m, inplace=True)  # using runopf wrapper


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Optimal Power Flow -- AC-polar-power formulation
MATPOWER Interior Point Solver -- MIPS, Version 1.5.2, 12-Jul-2025
 (using built-in linear solver)
Converged!
OPF successful

Converged in 0.25 seconds
Objective Function Value = 5296.69 $/hr
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    318.3              -9.6
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)    

In [ ]:
mpc.keys()

dict_keys(['version', 'baseMVA', 'bus', 'gen', 'branch', 'gencost'])

In [ ]:
print(m.workspace["mpc_"])
try:
    print(m.workspace["r1_"])
except TypeError as e:
    print(e)

{'version': '2', 'baseMVA': 100.0, 'bus': matlab.double([[1.0,3.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[2.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[3.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[4.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[5.0,1.0,90.0,30.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[6.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[7.0,1.0,100.0,35.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[8.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9],[9.0,1.0,125.0,50.0,0.0,0.0,1.0,1.0,0.0,345.0,1.0,1.1,0.9]]), 'gen': matlab.double([[1.0,72.3,27.03,300.0,-300.0,1.04,100.0,1.0,250.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[2.0,163.0,6.54,300.0,-300.0,1.025,100.0,1.0,300.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[3.0,85.0,-10.95,300.0,-300.0,1.025,100.0,1.0,270.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]]), 'branch': matlab.double([[1.0,4.0,0.0,0.0576,0.0,250.0,250.0,250.0,0.0,0.0,1.0,-360.0,360.0],[4.0,5.